### Transform Sprints Data

1. Read bronze_sprints table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
4. Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)
5. Filter out rows where season, round, constructor_id, or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of column race_name to Title Case
8. Write the transformed data to silver_sprints table

In [0]:
%run ../00-common/01.environment-configuration

In [0]:
bronze_table = F"{catalog_name}.{bronze_schema}.sprints"
silver_table = F"{catalog_name}.{silver_schema}.sprints"

In [0]:
sprints_df = spark.read.table(bronze_table)

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_selected_df = (
    sprints_df.select(
        F.col("date"),
        F.col("raceName"),
        F.col("round"),
        F.col("season"),
        F.col("constructorId"),
        F.col("driverId"),
        F.col("grid"),
        F.col("laps"),
        F.col("number"),
        F.col("points"),
        F.col("position"),
        F.col("positionText"),
        F.col("status"),
        F.col("timestamp"),
        F.col("source_file")
    )    
)

### Standardise column names using snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
### Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)

In [0]:
sprints_renamed_df = (
    sprints_selected_df
        .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "positionText": "finish_position_text",
            "date": "race_date",           
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "finish_position"
        })
)

In [0]:
display(sprints_renamed_df)

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,timestamp,source_file
2021-07-18,british grand prix,10,2021,red_bull,max_verstappen,2,17,33,3.0,1,1,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,mercedes,hamilton,1,17,44,2.0,2,2,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,mercedes,bottas,3,17,77,1.0,3,3,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,ferrari,leclerc,4,17,16,0.0,4,4,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,mclaren,norris,6,17,4,0.0,5,5,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,mclaren,ricciardo,7,17,3,0.0,6,6,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,alpine,alonso,11,17,14,0.0,7,7,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,aston_martin,vettel,10,17,5,0.0,8,8,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,williams,russell,8,17,63,0.0,9,9,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,alpine,ocon,13,17,31,0.0,10,10,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json


### Filter out rows where season, round, constructor_id, or driver_id is null (business key validation)

In [0]:
sprints_valid_df =(
    sprints_renamed_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )    
)        

In [0]:
display(sprints_valid_df)

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,timestamp,source_file
2021-07-18,british grand prix,10,2021,red_bull,max_verstappen,2,17,33,3.0,1,1,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,mercedes,hamilton,1,17,44,2.0,2,2,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,mercedes,bottas,3,17,77,1.0,3,3,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,ferrari,leclerc,4,17,16,0.0,4,4,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,mclaren,norris,6,17,4,0.0,5,5,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,mclaren,ricciardo,7,17,3,0.0,6,6,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,alpine,alonso,11,17,14,0.0,7,7,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,aston_martin,vettel,10,17,5,0.0,8,8,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,williams,russell,8,17,63,0.0,9,9,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,british grand prix,10,2021,alpine,ocon,13,17,31,0.0,10,10,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json


In [0]:
display(sprints_renamed_df.count()-sprints_valid_df.count())

7

### Remove duplicate records

In [0]:
sprints_distinct_df = (
    sprints_valid_df
        .dropDuplicates(["season", "round", "driver_id", "constructor_id"])
)

In [0]:
display(sprints_valid_df.count()-sprints_distinct_df.count())

13

### Transform values of column race_name to Title Case

In [0]:
sprints_final_df = (
    sprints_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(sprints_final_df)

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,timestamp,source_file
2021-07-18,British Grand Prix,10,2021,alpine,alonso,11,17,14,0.0,7,7,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mercedes,bottas,3,17,77,1.0,3,3,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,alphatauri,gasly,12,17,10,0.0,12,12,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,alfa,giovinazzi,14,17,99,0.0,15,15,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mercedes,hamilton,1,17,44,2.0,2,2,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,williams,latifi,18,17,6,0.0,17,17,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,ferrari,leclerc,4,17,16,0.0,4,4,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,red_bull,max_verstappen,2,17,33,3.0,1,1,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,haas,mazepin,20,17,9,0.0,19,19,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,haas,mick_schumacher,19,17,47,0.0,18,18,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json


In [0]:
(
    sprints_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))


race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,timestamp,source_file
2021-07-18,British Grand Prix,10,2021,alpine,alonso,11,17,14,0.0,7,7,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mercedes,bottas,3,17,77,1.0,3,3,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,alphatauri,gasly,12,17,10,0.0,12,12,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,alfa,giovinazzi,14,17,99,0.0,15,15,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,mercedes,hamilton,1,17,44,2.0,2,2,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,williams,latifi,18,17,6,0.0,17,17,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,ferrari,leclerc,4,17,16,0.0,4,4,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,red_bull,max_verstappen,2,17,33,3.0,1,1,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,haas,mazepin,20,17,9,0.0,19,19,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
2021-07-18,British Grand Prix,10,2021,haas,mick_schumacher,19,17,47,0.0,18,18,Finished,2026-09-13T19:20:29.459545Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2021.json
